# Logistic Regression - Hybrid GA + Local Random Search

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import numpy as np
import json
import copy
import time
import random
from data_loader import load_clinc
from ga_optimizer import GeneticOptimizer

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

USE_SAMPLE = False
data = load_clinc(use_sample=USE_SAMPLE)
train_texts, train_labels = data['train_texts'], data['train_labels']
val_texts, val_labels = data['val_texts'], data['val_labels']
test_texts, test_labels = data['test_texts'], data['test_labels']
num_classes = data['num_classes']

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.95)
train_features = vectorizer.fit_transform(train_texts).toarray()
val_features = vectorizer.transform(val_texts).toarray()
test_features = vectorizer.transform(test_texts).toarray()
print(f"TF-IDF dim: {train_features.shape[1]}")

cuda
15250 / 3100 / 5500, 151 classes (full)
TF-IDF dim: 5000


In [2]:
class TextDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


class LogisticRegression(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.linear(x)

In [3]:
PATIENCE = 5
MAX_EPOCHS = 50


def evaluate_hyperparams(genes):
    torch.manual_seed(42)
    np.random.seed(42)

    tr_loader = DataLoader(TextDataset(train_features, train_labels), batch_size=genes['batch_size'], shuffle=True)
    vl_loader = DataLoader(TextDataset(val_features, val_labels), batch_size=genes['batch_size'], shuffle=False)

    model = LogisticRegression(train_features.shape[1], num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=genes['learning_rate'], weight_decay=genes['weight_decay'])

    best_f1, patience_counter = 0, 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        for features, labels in tr_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(features), labels)
            loss.backward()
            optimizer.step()

        model.eval()
        all_preds = []
        with torch.no_grad():
            for features, labels in vl_loader:
                all_preds.extend(model(features.to(device)).argmax(1).cpu().numpy())

        vl_f1 = f1_score(val_labels, all_preds, average='macro', zero_division=0)

        if vl_f1 > best_f1:
            best_f1, patience_counter = vl_f1, 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            break

    return best_f1, {"val_f1": round(best_f1, 4), "epochs": epoch + 1}

## Phase 1: Genetic Algorithm

In [4]:
search_space = {
    'learning_rate': [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05],
    'weight_decay': [0, 1e-5, 1e-4, 1e-3],
    'batch_size': [32, 64, 128, 256]
}

TOTAL_BUDGET = 96
GA_BUDGET_FRACTION = 0.65
SEEDS = [42, 123, 456, 789, 1024]
all_hybrid_runs = []

for run_idx, seed in enumerate(SEEDS):
    print(f"Hybrid Run {run_idx+1}/5 (seed={seed})")

    ga = GeneticOptimizer(
        search_space=search_space,
        fitness_fn=evaluate_hyperparams,
        population_size=8,
        n_generations=9,
        crossover_rate=0.8,
        mutation_rate=0.2,
        tournament_size=3,
        elitism_count=2,
        seed=seed
    )
    ga_results = ga.run()
    ga_best = ga_results['best_genes']
    ga_evals = ga_results['total_evaluations']
    print(f"  Phase 1 (GA): Best F1={ga_results['best_fitness']:.4f}, Evals={ga_evals}")

    local_budget = TOTAL_BUDGET - ga_evals

    def build_neighborhood(search_space, best_genes, radius=1):
        neighborhood = {}
        for param, values in search_space.items():
            idx = values.index(best_genes[param])
            lo = max(0, idx - radius)
            hi = min(len(values) - 1, idx + radius)
            neighborhood[param] = values[lo:hi + 1]
        return neighborhood

    neighborhood = build_neighborhood(search_space, ga_best)

    random.seed(seed)
    local_results = []
    best_local_f1, best_local_params = ga_results['best_fitness'], ga_best
    local_start = time.time()

    for i in range(local_budget):
        params = {k: random.choice(v) for k, v in neighborhood.items()}
        eval_start = time.time()
        f1, metrics = evaluate_hyperparams(params)
        eval_time = time.time() - eval_start
        local_results.append({"params": params, "val_f1": round(f1, 4),
                              "eval_time": round(eval_time, 2)})
        if f1 > best_local_f1:
            best_local_f1, best_local_params = f1, params

    local_time = time.time() - local_start

    hybrid_best_f1 = best_local_f1
    hybrid_best_params = best_local_params
    total_evals = ga_evals + local_budget

    print(f"  Phase 2 (Local RS): {local_budget} evals in neighborhood")
    print(f"  Hybrid Best F1: {hybrid_best_f1:.4f}, Total evals: {total_evals}")

    all_hybrid_runs.append({
        "seed": seed,
        "best_fitness": hybrid_best_f1,
        "best_params": hybrid_best_params,
        "ga_best_fitness": ga_results['best_fitness'],
        "ga_best_params": ga_best,
        "ga_evals": ga_evals,
        "local_evals": local_budget,
        "total_evals": total_evals,
        "ga_time": ga_results['total_time_seconds'],
        "local_time": round(local_time, 2),
        "neighborhood": {k: [str(x) for x in v] for k, v in neighborhood.items()},
        "ga_history": ga_results['history'],
        "ga_all_evaluations": ga.all_evaluations,
        "local_results": local_results
    })

hybrid_val_f1s = [r['best_fitness'] for r in all_hybrid_runs]
print(f"\nHybrid val F1 (5 runs): {np.mean(hybrid_val_f1s):.4f} ± {np.std(hybrid_val_f1s):.4f}")

Hybrid Run 1/5 (seed=42)
Gen  1/9  best=0.8945  avg=0.8327  global_best=0.8945
Gen  2/9  best=0.8956  avg=0.8847  global_best=0.8956
Gen  3/9  best=0.8956  avg=0.8783  global_best=0.8956
Gen  4/9  best=0.8964  avg=0.8645  global_best=0.8964
Gen  5/9  best=0.8964  avg=0.8959  global_best=0.8964
Gen  6/9  best=0.8964  avg=0.8642  global_best=0.8964
Gen  7/9  best=0.8964  avg=0.8880  global_best=0.8964
Gen  8/9  best=0.8964  avg=0.8905  global_best=0.8964
Gen  9/9  best=0.8964  avg=0.8623  global_best=0.8964
  Phase 1 (GA): Best F1=0.8964, Evals=55
  Phase 2 (Local RS): 41 evals in neighborhood
  Hybrid Best F1: 0.8964, Total evals: 96
Hybrid Run 2/5 (seed=123)
Gen  1/9  best=0.8928  avg=0.7464  global_best=0.8928
Gen  2/9  best=0.8928  avg=0.8863  global_best=0.8928
Gen  3/9  best=0.8964  avg=0.8910  global_best=0.8964
Gen  4/9  best=0.8964  avg=0.8943  global_best=0.8964
Gen  5/9  best=0.8964  avg=0.8004  global_best=0.8964
Gen  6/9  best=0.8964  avg=0.8537  global_best=0.8964
Gen  7/9 

## Test Evaluation

In [5]:
def train_and_test(params):
    torch.manual_seed(42)
    np.random.seed(42)

    tr_loader = DataLoader(TextDataset(train_features, train_labels), batch_size=params['batch_size'], shuffle=True)
    vl_loader = DataLoader(TextDataset(val_features, val_labels), batch_size=params['batch_size'], shuffle=False)
    te_loader = DataLoader(TextDataset(test_features, test_labels), batch_size=params['batch_size'], shuffle=False)

    model = LogisticRegression(train_features.shape[1], num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=params['learning_rate'], weight_decay=params['weight_decay'])

    best_f1, best_state, patience_counter = 0, copy.deepcopy(model.state_dict()), 0
    for _ in range(MAX_EPOCHS):
        model.train()
        for features, labels in tr_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(features), labels)
            loss.backward()
            optimizer.step()

        model.eval()
        vl_preds = []
        with torch.no_grad():
            for features, labels in vl_loader:
                vl_preds.extend(model(features.to(device)).argmax(1).cpu().numpy())

        vl_f1 = f1_score(val_labels, vl_preds, average='macro', zero_division=0)
        if vl_f1 > best_f1:
            best_f1, best_state, patience_counter = vl_f1, copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1
        if patience_counter >= PATIENCE:
            break

    model.load_state_dict(best_state)
    model.eval()
    te_preds = []
    with torch.no_grad():
        for features, labels in te_loader:
            te_preds.extend(model(features.to(device)).argmax(1).cpu().numpy())

    metrics = {
        "accuracy": round(accuracy_score(test_labels, te_preds), 4),
        "macro_precision": round(precision_score(test_labels, te_preds, average='macro', zero_division=0), 4),
        "macro_recall": round(recall_score(test_labels, te_preds, average='macro', zero_division=0), 4),
        "macro_f1": round(f1_score(test_labels, te_preds, average='macro', zero_division=0), 4),
        "weighted_f1": round(f1_score(test_labels, te_preds, average='weighted', zero_division=0), 4),
    }
    return metrics, [int(x) for x in te_preds]


hybrid_test_results = []
for run in all_hybrid_runs:
    metrics, preds = train_and_test(run['best_params'])
    hybrid_test_results.append({"seed": run['seed'], "params": run['best_params'],
                                "metrics": metrics, "predictions": preds})
    print(f"  seed={run['seed']}: test F1={metrics['macro_f1']}")

hybrid_test_f1s = [r['metrics']['macro_f1'] for r in hybrid_test_results]
print(f"\nHybrid test F1: {np.mean(hybrid_test_f1s):.4f} ± {np.std(hybrid_test_f1s):.4f}")

  seed=42: test F1=0.8576
  seed=123: test F1=0.8576
  seed=456: test F1=0.8557
  seed=789: test F1=0.8576
  seed=1024: test F1=0.8557

Hybrid test F1: 0.8568 ± 0.0009


## Save Results

In [6]:
results = {
    "model_type": "LogisticRegression",
    "optimization": "hybrid_ga_local_rs",
    "n_runs": 5,
    "seeds": SEEDS,
    "total_budget": TOTAL_BUDGET,
    "val_f1_mean": round(np.mean(hybrid_val_f1s), 4),
    "val_f1_std": round(np.std(hybrid_val_f1s), 4),
    "test_f1_mean": round(np.mean(hybrid_test_f1s), 4),
    "test_f1_std": round(np.std(hybrid_test_f1s), 4),
    "ga_params": {
        "population_size": 8,
        "n_generations": 9,
        "crossover_rate": 0.8,
        "mutation_rate": 0.2,
        "tournament_size": 3,
        "elitism_count": 2
    },
    "runs": [{
        "seed": r['seed'],
        "best_hyperparameters": r['best_params'],
        "val_f1": round(r['best_fitness'], 4),
        "ga_best_val_f1": round(r['ga_best_fitness'], 4),
        "ga_evals": r['ga_evals'],
        "local_evals": r['local_evals'],
        "total_evals": r['total_evals'],
        "ga_time_seconds": r['ga_time'],
        "local_time_seconds": r['local_time'],
        "test_metrics": hybrid_test_results[i]['metrics'],
        "predictions": hybrid_test_results[i]['predictions'],
        "neighborhood": r['neighborhood'],
        "ga_history": r['ga_history'],
        "ga_all_evaluations": r['ga_all_evaluations'],
        "local_results": r['local_results']
    } for i, r in enumerate(all_hybrid_runs)]
}

with open('results/results_lr_hybrid.json', 'w') as f:
    json.dump(results, f, indent=4, default=str)

print("Saved: results/results_lr_hybrid.json")

Saved: results/results_lr_hybrid.json
